# FFTMAD → JAX: Variational Newton-CG for Phase-Field Fracture

**Goal**: Translate `variational_nw_cg_st_pff_small` from FFTMAD (NumPy + SciPy) into
modern JAX — JIT-compiled, GPU-ready, autodiff-enabled.

**Key changes vs. FFTMAD**:
| FFTMAD | JAX |
|--------|-----|
| `simu` object passed everywhere | pure functions + explicit state pytrees |
| `scipy.sparse.linalg.cg` + `LinearOperator` | `jax.scipy.sparse.linalg.cg` + plain function |
| `for` / `while` loops in Python | `jax.lax.while_loop` (compiled, no Python overhead) |
| NumPy arrays, mutable | JAX arrays, immutable (use `.at[].set()`) |
| CPU only, thread-level parallel | CPU / GPU / TPU via XLA |
| No autodiff through solver | `jax.grad` through full staggered loop |

References: Lucarini & Segurado (2019), arXiv:1905.12725 (DBFFT), arXiv:2304.01125 (FFT + PFF)

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap
from jax.scipy.sparse.linalg import cg as jax_cg
from functools import partial
import numpy as np

# enable 64-bit (matches FFTMAD float64 default)
jax.config.update('jax_enable_x64', True)

print('JAX version:', jax.__version__)
print('Default backend:', jax.default_backend())
print('Devices:', jax.devices())

## 1  Core Operators

### `ddot42`  —  4th-order : 2nd-order contraction

In FFTMAD:
```python
# shape convention: tensor indices first, voxel index last
# A: (3,3,3,3, N_vox)   B: (3,3, N_vox)  →  (3,3, N_vox)
def ddot42(A, B, simu):
    return np.einsum('ijklm,klm->ijm', A, B)
```
In JAX this is identical — just swap `np` for `jnp`:

In [ ]:
@jit
def ddot42(A, B):
    """C_ij = A_ijkl B_kl  (all shapes: (3,3,...,N) etc.)"""
    # A: (3,3,3,3,N)  B: (3,3,N) -> (3,3,N)
    return jnp.einsum('ijklm,klm->ijm', A, B)

@jit
def ddot42_nd(A, B):
    """Same but for arbitrary spatial shape: A (i,j,k,l,*spatial), B (k,l,*spatial)"""
    # reshape spatial dims into single N axis, apply, reshape back
    spatial = A.shape[4:]
    N = int(np.prod(spatial))
    A_r = A.reshape(3, 3, 3, 3, N)
    B_r = B.reshape(3, 3, N)
    return jnp.einsum('ijklm,klm->ijm', A_r, B_r).reshape(3, 3, *spatial)

# FFT helpers — wrap so spatial axes are explicit
@jit
def fft3(x):
    """FFT over last 3 axes (spatial). x: (..., N1, N2, N3)"""
    return jnp.fft.fftn(x, axes=(-3, -2, -1))

@jit
def ifft3(x):
    """IFFT over last 3 axes, return real part."""
    return jnp.fft.ifftn(x, axes=(-3, -2, -1)).real

# 2D versions for plane-problem notebooks
@jit
def fft2(x):   return jnp.fft.fftn(x, axes=(-2, -1))
@jit
def ifft2(x):  return jnp.fft.ifftn(x, axes=(-2, -1)).real

## 2  Green's Operator  $\hat{\boldsymbol{\Gamma}}_0(\boldsymbol{\xi})$

For an **isotropic reference medium** $(\lambda_0, \mu_0)$, the acoustic tensor is
$$K_{ij}(\boldsymbol{\xi}) = (\lambda_0+\mu_0)\xi_i\xi_j + \mu_0|\boldsymbol{\xi}|^2\delta_{ij}$$

with closed-form inverse
$$K^{-1}_{ij} = \frac{1}{\mu_0|\boldsymbol{\xi}|^2}\left(\delta_{ij} - \frac{\lambda_0+\mu_0}{\lambda_0+2\mu_0}\hat{n}_i\hat{n}_j\right)$$

The strain Green's operator then reads
$$\hat{\Gamma}_{0,ijkl} = -\tfrac{1}{4}\left(K^{-1}_{ik}\hat{n}_j\hat{n}_l + K^{-1}_{il}\hat{n}_j\hat{n}_k + K^{-1}_{jk}\hat{n}_i\hat{n}_l + K^{-1}_{jl}\hat{n}_i\hat{n}_k\right)$$

At $\boldsymbol{\xi}=\mathbf{0}$: $\hat{\boldsymbol{\Gamma}}_0 = \mathbf{0}$ (mean strain is prescribed).

In [ ]:
def build_freq_grid(n, L):
    """Build flat (ndim, N_vox) frequency array from grid shape n and box size L."""
    freqs = [jnp.fft.fftfreq(ni, d=Li/ni) * 2 * jnp.pi
             for ni, Li in zip(n, L)]
    grids = jnp.meshgrid(*freqs, indexing='ij')   # each: (N1,N2,N3)
    xi_grid = jnp.stack([g.ravel() for g in grids])  # (ndim, N_vox)
    return xi_grid

def build_green_operator(xi_flat, lam0, mu0):
    """
    Build the isotropic Green's operator Γ̂₀ for all frequencies.
    xi_flat : (ndim, N_vox)  —  flattened frequency grid
    Returns  : (ndim, ndim, ndim, ndim, N_vox)
    """
    ndim, N = xi_flat.shape
    xi_sq   = jnp.sum(xi_flat**2, axis=0)          # (N,)
    safe    = xi_sq > 0                             # mask for ξ ≠ 0
    xi_sq_s = jnp.where(safe, xi_sq, 1.0)          # avoid /0

    # unit normal  n̂ = ξ / |ξ|
    n_hat   = xi_flat / jnp.sqrt(xi_sq_s)[None, :]  # (ndim, N)

    # K⁻¹_ij = (δ_ij - c * n̂_i n̂_j) / (μ₀ |ξ|²)
    c     = (lam0 + mu0) / (lam0 + 2*mu0)
    delta = jnp.eye(ndim)                           # (ndim, ndim)
    Kinv  = (delta[:, :, None]
             - c * jnp.einsum('iN,jN->ijN', n_hat, n_hat)
            ) / (mu0 * xi_sq_s[None, None, :])      # (ndim,ndim,N)

    # Γ̂₀_ijkl = -¼ (K⁻¹_ik n̂_j n̂_l + K⁻¹_il n̂_j n̂_k
    #                + K⁻¹_jk n̂_i n̂_l + K⁻¹_jl n̂_i n̂_k)
    nn = jnp.einsum('iN,jN->ijN', n_hat, n_hat)    # n̂_i n̂_j  (ndim,ndim,N)
    Gamma = -0.25 * (
        jnp.einsum('ikN,jlN->ijklN', Kinv, nn) +
        jnp.einsum('ilN,jkN->ijklN', Kinv, nn) +
        jnp.einsum('jkN,ilN->ijklN', Kinv, nn) +
        jnp.einsum('jlN,ikN->ijklN', Kinv, nn)
    )                                                # (ndim,ndim,ndim,ndim,N)

    # zero out ξ = 0 frequency (mean strain is prescribed separately)
    return jnp.where(safe[None, None, None, None, :], Gamma, 0.0)

## 3  Material Models

Elastic stiffness tensor $\mathbb{C}_0$ for an **isotropic** phase (Voigt symmetry):
$$C_{ijkl} = \lambda\,\delta_{ij}\delta_{kl} + \mu(\delta_{ik}\delta_{jl}+\delta_{il}\delta_{jk})$$

Damaged tangent: $\mathbb{C}_\text{tan}(d) = g(d)\,\mathbb{C}_0$ where $g(d) = (1-d)^2 + \kappa$.

In [ ]:
def isotropic_stiffness(lam, mu, ndim=3):
    """Build (ndim,ndim,ndim,ndim) isotropic 4th-order stiffness tensor."""
    delta = jnp.eye(ndim)
    C  = lam * jnp.einsum('ij,kl->ijkl', delta, delta)
    C += mu  * (jnp.einsum('ik,jl->ijkl', delta, delta)
              + jnp.einsum('il,jk->ijkl', delta, delta))
    return C                                          # (3,3,3,3)

def make_stiffness_field(phase_map, materials):
    """
    Build per-voxel stiffness field.
    phase_map : (N_vox,)  int   — material index per voxel
    materials : list of (3,3,3,3) tensors
    Returns   : (3,3,3,3,N_vox)
    """
    C_stack = jnp.stack(materials, axis=-1)           # (3,3,3,3, n_mat)
    return C_stack[:, :, :, :, phase_map]             # (3,3,3,3, N_vox)

# Degradation function g(d) = (1-d)^2 + κ
def degradation(d, kappa=1e-6):
    return (1.0 - d)**2 + kappa

def degradation_prime(d, kappa=1e-6):
    return -2.0 * (1.0 - d)

def damaged_tangent(C0_field, d_field):
    """
    C0_field : (3,3,3,3, N_vox)  — undamaged stiffness
    d_field  : (N_vox,)          — damage variable
    Returns  : (3,3,3,3, N_vox)  — degraded tangent
    """
    g = degradation(d_field)    # (N_vox,)
    return C0_field * g[None, None, None, None, :]

## 4  The A-Operator — Matrix-Free Tangent in JAX

**FFTMAD** (from `dstrain_loc_CPU_nw_CG2`):
```python
def A_operator(var):
    strain_inc = var.reshape(simu.shape2)
    tmp    = ddot42(simu.tangent_glob, strain_inc, simu)   # C_tan : δε
    tmp_ft = fft(tmp, simu)
    out_ft = ddot42(simu.G_glob, tmp_ft, simu)             # Γ̂₀ : ...
    return ifft(out_ft, simu).reshape(-1)
```

**JAX**: exactly the same logic — but the function is pure (no `simu`) and JIT-compiled:

In [ ]:
def make_A_operator(G_glob, C_tan_field, shape):
    """
    Returns a matrix-free linear operator  A: R^(9*N) -> R^(9*N)
    that applies:  A(δε) = IFFT[ Γ̂₀ : FFT[ C_tan : δε ] ]

    G_glob    : (3,3,3,3,N_vox)  — Green's operator in Fourier space (precomputed)
    C_tan     : (3,3,3,3,N_vox)  — current tangent stiffness (updated each Newton iter)
    shape     : (3,3,N_vox)       — strain field shape
    """
    @jit
    def A_op(var_flat):
        delta_eps = var_flat.reshape(shape)                # (3,3,N_vox)
        tmp       = ddot42(C_tan_field, delta_eps)         # C_tan : δε
        tmp_ft    = fft3(tmp)                              # FFT over spatial
        out_ft    = ddot42(G_glob, tmp_ft)                 # Γ̂₀ : ...
        return ifft3(out_ft).reshape(-1)                   # back to flat real
    return A_op

def build_rhs(G_glob, stress_res):
    """
    RHS of Newton system:
        b = -IFFT[ Γ̂₀ : FFT[ σ_loc - σ_goal ] ]

    stress_res : (3,3,N_vox)  — stress residual
    Returns    : (9*N_vox,)   — flat vector for CG
    """
    res_ft = fft3(stress_res)
    out_ft = ddot42(G_glob, res_ft)
    return -ifft3(out_ft).reshape(-1)

## 5  Newton-CG Step

**FFTMAD** uses `scipy.sparse.linalg.cg(LinearOperator(...), bb, tol=...)`.  
**JAX** has `jax.scipy.sparse.linalg.cg(A_fn, b, ...)` — takes a plain **callable**,
no `LinearOperator` wrapper needed. The whole CG loop runs inside a single
`jax.lax.while_loop`, so no Python overhead per CG iteration.

Adaptive forcing tolerance (same as FFTMAD):
$$\text{tol}^{(k)} = \max\!\left(\texttt{toler\_lin}\cdot\frac{\|b_0\|}{\|b^{(k)}\|},\;\texttt{toler\_lin}\right)$$

In [ ]:
@partial(jit, static_argnames=('maxiter_cg',))
def newton_cg_step(G_glob, C_tan_field, stress_loc, stress_goal,
                   strain_shape, bb0n, toler_lin, maxiter_cg=500):
    """
    One Newton-CG iteration — solve  A(δε) = b  for the strain increment.

    Returns
    -------
    delta_eps_flat : (9*N_vox,)  strain increment
    info           : int   0 = converged, >0 = max iter reached
    bb0n_new       : float  updated reference residual norm (first Newton iter)
    """
    # --- RHS: stress residual projected by Green's operator ---
    stress_res = stress_loc - stress_goal
    bb         = build_rhs(G_glob, stress_res)          # (9*N,)
    bbn        = jnp.linalg.norm(bb)

    # update reference norm on first Newton step (iter_nw==0 in FFTMAD)
    bb0n_new   = jnp.where(bb0n < 0, bbn, bb0n)        # <0 signals 'first iter'

    # adaptive CG tolerance (forcing condition)
    tol = jnp.maximum(toler_lin * bb0n_new / (bbn + 1e-300), toler_lin)

    # --- matrix-free operator ---
    A_op = make_A_operator(G_glob, C_tan_field, strain_shape)

    # --- CG solve (jax.scipy, runs inside lax.while_loop) ---
    x0 = jnp.zeros(bb.shape, dtype=bb.dtype)
    delta_eps_flat, info = jax_cg(A_op, bb, x0=x0,
                                   tol=tol, maxiter=maxiter_cg)
    return delta_eps_flat, info, bb0n_new


# Newton convergence check  (same criterion as FFTMAD)
@jit
def newton_error(delta_eps_flat, strain_loc):
    """errorNW = max|δε| / ‖avg(ε)‖"""
    return (jnp.max(jnp.abs(delta_eps_flat))
            / (jnp.linalg.norm(jnp.mean(strain_loc, axis=-1)) + 1e-300))

## 6  Helmholtz / Damage Solve

The phase-field (AT2) damage equation is **diagonal in Fourier space**:
$$\left[\frac{\mathcal{G}_c}{c_w\ell} + 2\mathcal{G}_c\ell\,|\boldsymbol{\xi}|^2 - g''(d)\mathcal{H}\right]\hat{d}(\boldsymbol{\xi}) = \hat{r}(\boldsymbol{\xi})$$

This is a single FFT + pointwise divide + IFFT — no iterative solver needed.

In FFTMAD this is wrapped in `helmholtz_test_precond` which additionally applies a
preconditioner and uses `scipy.sparse.linalg.cg` for the full non-linear case.
Here we show the **direct diagonal solve** first, then the CG version.

In [ ]:
@jit
def damage_step_direct(d_prev, H, xi_sq_flat, Gc, ell, cw, kappa=1e-6):
    """
    Direct (non-iterative) AT2 damage solve via diagonal Fourier inversion.

    AT2 strong form:
        2 Gc/(cw ell) d  -  2 Gc ell/cw  Δd  =  -g'(d) H  =  2(1-d) H

    In Fourier space this is diagonal:
        D(ξ) d̂(ξ) = r̂(ξ)

    D(ξ) = 2 Gc/cw * (1/ell + ell |ξ|²)   ← Fourier-space diagonal, no FFT of D
    r(x) = -g'(d) H = 2(1-d) H             ← real-space RHS, must be FFT'd

    d_prev     : (N_vox,)  previous damage
    H          : (N_vox,)  history variable  max_{s≤t} ψ⁺
    xi_sq_flat : (N_vox,)  |ξ|²  on flattened frequency grid (precomputed, Fourier-space)
    """
    # D lives in Fourier space already (ξ-dependent) — do NOT FFT it
    D = (2.0 * Gc / cw) * (1.0 / ell + ell * xi_sq_flat)   # (N_vox,)

    # RHS is a real-space field — must be FFT'd
    rhs_real = -degradation_prime(d_prev, kappa) * H         # = 2*(1-d)*H
    rhs_ft   = jnp.fft.fft(rhs_real)

    # diagonal solve in Fourier space, then back
    d_new = jnp.fft.ifft(rhs_ft / (D + 1e-300)).real

    # irreversibility  d ≥ d_prev,  d ∈ [0, 1]
    return jnp.clip(jnp.maximum(d_new, d_prev), 0.0, 1.0)


@partial(jit, static_argnames=('maxiter',))
def damage_step_cg(d_prev, H, xi_sq_flat, Gc, ell, cw,
                   kappa=1e-6, tol=1e-8, maxiter=200):
    """
    CG damage solve — mirrors FFTMAD's helmholtz_test_precond.
    Operator: A_H(d̂) = D(ξ) d̂(ξ)  applied via FFT → multiply → IFFT.
    Preconditioner M⁻¹: divide by D in Fourier space (diagonal precond).
    """
    D = (2.0 * Gc / cw) * (1.0 / ell + ell * xi_sq_flat)

    def A_helmholtz(d_flat):
        d_ft = jnp.fft.fft(d_flat)
        return jnp.fft.ifft(D * d_ft).real

    def M_inv(r_flat):                         # diagonal preconditioner
        r_ft = jnp.fft.fft(r_flat)
        return jnp.fft.ifft(r_ft / (D + 1e-300)).real

    rhs = -degradation_prime(d_prev, kappa) * H
    d_new, info = jax_cg(A_helmholtz, rhs, x0=d_prev,
                         M=M_inv, tol=tol, maxiter=maxiter)
    return jnp.clip(jnp.maximum(d_new, d_prev), 0.0, 1.0), info

## 7  Full Staggered Loop with `jax.lax.while_loop`

The staggered (fixed-point) loop in FFTMAD is a Python `while` loop:
```python
while errorST > toler_st and iter_st < maxiter_st:
    # Newton-CG (mechanics)
    # Helmholtz (damage)
    errorST = max(errM, errC1)
```

In JAX, `jax.lax.while_loop(cond_fn, body_fn, init_state)` compiles this entire loop
into a **single XLA kernel** — no Python overhead per iteration.

State pytree passed through the loop:

In [ ]:
from typing import NamedTuple

class StaggeredState(NamedTuple):
    strain_loc : jnp.ndarray   # (3,3,N_vox)
    stress_loc : jnp.ndarray   # (3,3,N_vox)
    dam        : jnp.ndarray   # (N_vox,)
    H          : jnp.ndarray   # (N_vox,)
    error_st   : jnp.ndarray   # scalar  — must be jnp array, not Python float
    iter_st    : jnp.ndarray   # scalar int — must be jnp array for lax.while_loop


def make_staggered_step(G_glob, C0_field, xi_sq_flat,
                        stress_goal, Gc, ell, cw,
                        toler_nw, toler_lin, maxiter_nw, maxiter_cg):
    """
    Returns a jit-compiled staggered body function.
    All static parameters are closed over so nothing leaks into lax.while_loop state.
    """

    @jit
    def constitutive_update(strain_loc, dam):
        C_tan    = damaged_tangent(C0_field, dam)
        stress   = ddot42(C_tan, strain_loc)
        psi_plus = 0.5 * jnp.einsum('ijm,ijm->m', stress, strain_loc)
        return stress, C_tan, psi_plus

    # Newton state: (strain_loc, dam, error_nw, bb0n)
    # dam is fixed during mechanics — carried so constitutive_update can use it
    @jit
    def newton_body(nw_state):
        strain_loc, dam, error_nw, bb0n = nw_state
        stress_loc, C_tan, _ = constitutive_update(strain_loc, dam)
        delta_flat, _, bb0n  = newton_cg_step(
            G_glob, C_tan, stress_loc, stress_goal,
            strain_loc.shape, bb0n, toler_lin, maxiter_cg)
        strain_loc = strain_loc + delta_flat.reshape(strain_loc.shape)
        error_nw   = newton_error(delta_flat, strain_loc)
        return strain_loc, dam, error_nw, bb0n

    @jit
    def staggered_body(state: StaggeredState) -> StaggeredState:
        dam_k = state.dam

        # Newton-CG loop — dam fixed, strain updated
        init_nw = (state.strain_loc, state.dam,
                   jnp.array(1.0), jnp.array(-1.0))   # bb0n=-1 → first-iter flag

        strain_loc, _, _, _ = jax.lax.while_loop(
            cond_fun=lambda s: s[2] > toler_nw,
            body_fun=newton_body,
            init_val=init_nw,
        )

        # final constitutive update at converged strain
        stress_loc, _, psi_plus = constitutive_update(strain_loc, state.dam)

        # history variable: running max of ψ⁺
        H_new = jnp.maximum(state.H, psi_plus)

        # damage solve
        dam_new = damage_step_direct(state.dam, H_new, xi_sq_flat, Gc, ell, cw)

        # staggered error: relative change in damage
        dam_norm = jnp.linalg.norm(dam_new)
        err_d = jnp.where(
            dam_norm > 1e-10,
            jnp.linalg.norm(dam_new - dam_k) / dam_norm,
            jnp.linalg.norm(dam_new - dam_k),
        )

        return StaggeredState(
            strain_loc = strain_loc,
            stress_loc = stress_loc,
            dam        = dam_new,
            H          = H_new,
            error_st   = err_d,
            iter_st    = state.iter_st + 1,
        )

    return staggered_body


# staggered_body_fn is a Python callable (jit-compiled function).
# JAX cannot trace it as an abstract array — it must be static.
@partial(jit, static_argnames=('staggered_body_fn', 'maxiter_st'))
def run_staggered(init_state, staggered_body_fn, toler_st, maxiter_st):
    """Run the staggered loop until convergence or maxiter_st reached."""
    return jax.lax.while_loop(
        cond_fun=lambda s: (s.error_st > toler_st) & (s.iter_st < maxiter_st),
        body_fun=staggered_body_fn,
        init_val=init_state,
    )

## 8  Autodiff Through the Solver

Because every operation above is JAX-traceable, `jax.grad` differentiates
**through the full staggered + Newton-CG loop** — including the `lax.while_loop`.

Use cases for phase-field fracture:
- Calibrate $\ell$ or $\mathcal{G}_c$ from DIC crack-path data
- Optimize fiber direction $\mathbf{a}_0$ for maximum fracture resistance  
- Sensitivity of homogenized toughness to microstructure volume fraction

In [ ]:
# Example: gradient of total fracture energy w.r.t. regularization length ℓ

def total_fracture_energy(ell, init_state, staggered_body_fn,
                           xi_sq_flat, Gc, cw, toler_st=1e-6, maxiter_st=50):
    """Run simulation and return scalar fracture energy — differentiable in ell."""
    final = run_staggered(init_state, staggered_body_fn, toler_st, maxiter_st)
    # crack density functional:  E_frac = ∫ Gc * Γ_ell(d, ∇d) dV
    # simplified: E ≈ Gc/cw * (sum(d²)/ell + ell * sum(|∇d|²)) * dV
    d = final.dam
    d_ft = jnp.fft.fft(d)
    grad_d_sq = jnp.sum(xi_sq_flat * jnp.abs(d_ft)**2) / d.size
    return Gc / cw * (jnp.sum(d**2) / ell + ell * grad_d_sq)

# dE/dℓ via reverse-mode autodiff through the solver
# dE_dl = jax.grad(total_fracture_energy)(ell_init, ...)

# HVP via forward-over-reverse (for Newton-CG on the outer optimisation)
# hvp = lambda v: jax.jvp(jax.grad(total_fracture_energy), (ell,), (v,))[1]

## 9  Mini Demo — Single-Phase Elastic + Damage RVE

In [ ]:
# ---- problem setup ----
n  = (32, 32, 1)      # 2D plane-problem: Nz=1
L  = (1.0, 1.0, 1.0)  # box size [mm]
dV = np.prod(L) / np.prod(n)

# material: E=200 GPa, ν=0.3  →  λ, μ
E, nu = 200e3, 0.3    # [MPa]
lam   = E*nu / ((1+nu)*(1-2*nu))
mu    = E / (2*(1+nu))
C0    = isotropic_stiffness(lam, mu, ndim=3)

# homogeneous microstructure (single phase)
N_vox     = int(np.prod(n))
phase_map = jnp.zeros(N_vox, dtype=int)
C0_field  = make_stiffness_field(phase_map, [C0])

# reference medium = material itself (linear elastic → converges in 1 step)
lam0, mu0 = lam, mu

# frequency grid
xi_flat  = build_freq_grid(n, L)          # (3, N_vox)
xi_sq    = jnp.sum(xi_flat**2, axis=0)    # (N_vox,)
G_glob   = build_green_operator(xi_flat, lam0, mu0)  # (3,3,3,3,N_vox)

# PFF parameters
Gc  = 2.7e-3   # [N/mm]  (glass-fiber level)
ell = 0.05     # [mm]    regularization length
cw  = 0.5      # AT2 normalization constant

# prescribed macroscopic strain (uniaxial tension)
eps_bar      = jnp.zeros((3, 3))
eps_bar      = eps_bar.at[0, 0].set(1e-4)   # ε_11 = 1e-4
stress_goal  = jnp.zeros((3, 3, N_vox))     # stress-free reference

# initial state
strain_init  = jnp.ones((3, 3, N_vox)) * eps_bar[:, :, None]
stress_init  = ddot42(C0_field, strain_init)
dam_init     = jnp.zeros(N_vox)
H_init       = jnp.zeros(N_vox)

init_state = StaggeredState(
    strain_loc = strain_init,
    stress_loc = stress_init,
    dam        = dam_init,
    H          = H_init,
    error_st   = jnp.array(1.0),
    iter_st    = jnp.array(0),
)

print(f'Grid: {n}  |  N_vox: {N_vox}')
print(f'λ={lam:.1f}, μ={mu:.1f} MPa  |  Gc={Gc}, ℓ={ell} mm')

In [ ]:
import time

# JIT-compile by running once (tracing)
staggered_body = make_staggered_step(
    G_glob, C0_field, xi_sq,
    stress_goal, Gc, ell, cw,
    toler_nw=1e-8, toler_lin=1e-8,
    maxiter_nw=20, maxiter_cg=500)

print('Tracing (first call — JIT compile)...')
t0 = time.perf_counter()
final = run_staggered(init_state, staggered_body, toler_st=1e-6, maxiter_st=50)
jax.block_until_ready(final)
t_compile = time.perf_counter() - t0

print('Second call (compiled kernel)...')
t0 = time.perf_counter()
final = run_staggered(init_state, staggered_body, toler_st=1e-6, maxiter_st=50)
jax.block_until_ready(final)
t_run = time.perf_counter() - t0

print(f'Compile: {t_compile:.3f}s  |  Run: {t_run*1000:.2f}ms')
print(f'Staggered iters: {int(final.iter_st)}')
print(f'Max damage: {float(jnp.max(final.dam)):.6f}')

## 10  Notes on Full FFTMAD Port

### What still needs to be done

| FFTMAD feature | JAX translation needed |
|----------------|------------------------|
| `eval_FP` / UMAT interface | wrap Fortran UMAT via `jax.pure_callback` or rewrite in JAX |
| `pnewdt` failure + time-step reduction | carry a `failed: bool` flag through `lax.while_loop`, use `lax.cond` |
| Mixed strain/stress control | replace $\hat{\mathbf{G}}_\text{glob}[:,0]$ with `lax.select` per frequency |
| Willot rotated grid | replace `jnp.fft.fftfreq` with Willot discrete derivative in `build_green_operator` |
| `helmholtz_test_precond` preconditioner | diagonal Fourier preconditioner: `M⁻¹(x) = IFFT(x̂ / D̂)` |
| ParaView output | extract every N steps with `jax.lax.cond`, convert via `jnp.ndarray.tofile` |
| Interactive plotting | `jax.experimental.io_callback` for matplotlib calls |

### Willot discrete derivative (drop-in for `build_green_operator`)

```python
def willot_deriv(xi, h):
    """Willot (2015) rotated staggered-grid derivative operator."""
    # xi: (ndim, N_vox)  h: (ndim,) voxel sizes
    # ∂̂_i^W = (2/h_i) sin(ξ_i h_i/2) exp(i ξ_i h_i/2)
    xi_h = xi * h[:, None] / 2
    return 2.0 / h[:, None] * jnp.sin(xi_h) * jnp.exp(1j * xi_h)
```
Pass `willot_deriv(xi, h)` instead of `xi` into `build_green_operator`.